# Root Cause Analysis

1. Aggregate transaction-level data.
To make the analysis more meaningful, I introduced controlled perturbations to simulate noticeable growth patterns.

2. Select a target merchant and benchmark key business metrics (e.g., transaction amount, transaction count) against peer merchants within the same category.

In [1]:
import pandas as pd
import numpy as np
from helper import get_generation

In [2]:
# Load dataset from huggingface
splits = {'train': 'credit_card_transaction_train.csv', 'test': 'credit_card_transaction_test.csv'}
df_all = pd.read_csv("hf://datasets/pointe77/credit-card-transaction/" + splits["train"], index_col=0)

/Users/yanxinye/github/ml-ops-notes/ml/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ETL
df_all = df_all.assign(
    trans_date_trans_time=lambda x: pd.to_datetime(x['trans_date_trans_time']),
    trans_date=lambda x: x['trans_date_trans_time'].dt.strftime('%Y-%m-%d')
)

In [4]:
TARGET_MERCHANTS = "fraud_Kilback LLC"

# Date range of the analysis
START_DATE_TY = "2020-01-01"
START_DATE_LY = "2019-01-01"
END_DATE_TY = "2020-05-31"
END_DATE_LY = "2019-05-31"

In [6]:
df = df_all[
    (
        (df_all['trans_date'] >= START_DATE_TY) &
        (df_all['trans_date'] <= END_DATE_TY)
    )
    |
    (
        (df_all['trans_date'] >= START_DATE_LY) &
        (df_all['trans_date'] <= END_DATE_LY)
    )
].copy()

In [7]:
# Convert dob to datetime and extract generation
df = df.assign(
    dob=lambda x: pd.to_datetime(x['dob']),
    generation=lambda x: x['dob'].dt.year.map(get_generation)
)

In [8]:
df_comparison = (
    df.assign(
        yr_flag=lambda x: x['trans_date'].apply(
            lambda d: 'TY' if START_DATE_TY <= d <= END_DATE_TY
            else ('LY' if START_DATE_LY <= d <= END_DATE_LY else None)
        )
    )
    .groupby(['generation', 'state', 'gender', 'yr_flag', 'category'], as_index=False)['amt'].sum()
    .pivot_table(
        index=['generation', 'state', 'gender', 'category'],
        columns='yr_flag',
        values='amt',
        fill_value=0
    )
    .reset_index()
    .rename(columns={'TY': 'amt_ty', 'LY': 'amt_ly'})
)





In [9]:
# This data does not have any growth. Add some random variation to make it more interesting.

# Add more variation to amt_ty and amt_ly for bigger growth

np.random.seed(170)
random_factors_ty = np.clip(np.random.normal(0, 0.5, len(df_comparison)), -1, 1)
random_factors_ly = np.clip(np.random.normal(0, 0.3, len(df_comparison)), -0.5, 0.5)

df_comparison['amt_ty'] *= (1 + random_factors_ty)

df_comparison['amt_ly'] *= (1 + random_factors_ly)

In [10]:

df_comparison['amt_diff'] = df_comparison['amt_ty'] - df_comparison['amt_ly']
total_amt_diff = df_comparison['amt_diff'].sum()

df_comparison['amt_growth'] = (
    (df_comparison['amt_ty'] - df_comparison['amt_ly']) 
    / df_comparison['amt_ly'].replace(0, pd.NA)
)

total_amt_ly = df_comparison['amt_ly'].sum()
print(f"Total amount in LY: {total_amt_ly}")
print(f"Total amount difference: {total_amt_diff}")
print(f"Total growth: {total_amt_diff / total_amt_ly:.2%}")
df_comparison["amt_ctc"] = 1/total_amt_ly * df_comparison["amt_diff"]


df_comparison

Total amount in LY: 22170636.606175352
Total amount difference: -483637.2378439163
Total growth: -2.18%


yr_flag,generation,state,gender,category,amt_ly,amt_ty,amt_diff,amt_growth,amt_ctc
0,Boomer,AL,F,entertainment,4165.903926,4841.668488,675.764562,0.162213,3.048016e-05
1,Boomer,AL,F,food_dining,5847.228107,867.420070,-4979.808037,-0.851653,-2.246128e-04
2,Boomer,AL,F,gas_transport,10188.932949,11721.189077,1532.256129,0.150384,6.911196e-05
3,Boomer,AL,F,grocery_net,842.792565,2471.100000,1628.307435,1.932038,7.344432e-05
4,Boomer,AL,F,grocery_pos,9495.259146,9500.454047,5.194901,0.000547,2.343145e-07
...,...,...,...,...,...,...,...,...,...
4789,Silent,WY,F,misc_pos,186.891003,0.000000,-186.891003,-1.0,-8.429663e-06
4790,Silent,WY,F,personal_care,149.854856,764.848705,614.993849,4.10393,2.773912e-05
4791,Silent,WY,F,shopping_net,71.655000,894.836931,823.181931,11.48813,3.712938e-05
4792,Silent,WY,F,shopping_pos,59.070000,66.591199,7.521199,0.127327,3.392414e-07


In [11]:
df_comparison["amt_ctc"].sum()

np.float64(-0.021814314421138677)